In [1]:
import os, glob
import mdtraj as md
from openmm import *
from openmm.app import *
from openmm.unit import *

In [2]:
pdb_fns = sorted(glob.glob('./for_Jo/*.pdb') + glob.glob('./Talha_Compounds/*.pdb'))
sys_fns = [fn.replace('.pdb', '_sys.xml') for fn in pdb_fns]

In [5]:
#Scale XH1 XH2 XH3 XH4 by these factors
dH_per_H = [element.hydrogen.mass * scale - element.hydrogen.mass for scale in [2, 2.5, 2.5, 2.5]]

for pdb_fn, sys_fn in zip(pdb_fns, sys_fns):
    assert False not in [os.path.isfile(pdb_fn), os.path.isfile(sys_fn)]
    print(pdb_fn, sys_fn)
    #Load
    top = PDBFile(pdb_fn).topology
    with open(sys_fn, 'r') as f:
        sys = XmlSerializer.deserialize(f.read())
    #Change the force groups in case you want MTS
    for force in sys.getForces():
        if type(force) != NonbondedForce:
            force.setForceGroup(0)
            #print(force, type(force), 0)
        else:
            force.setForceGroup(0)
            force.setReciprocalSpaceForceGroup(1)
            #print(force, type(force), 1)
    
    for force in sys.getForces():
        print(force, force.getForceGroup())

    with open(sys_fn.replace('_sys.xml', '_FG_sys.xml'), 'w') as f:
        f.write(XmlSerializer.serialize(sys))

    #Adjust Masses for HMR
    mass_current = [sys.getParticleMass(i) for i in range(sys.getNumParticles())]
    not_water_indices = md.load(pdb_fn).top.select('not water')

    #Print Mass before
    mass_before = sum([sys.getParticleMass(i) for i in range(sys.getNumParticles())])
    
    covalent_H_bonds = []
    
    for bond in top.bonds():
        is_h = [element.hydrogen == atom.element for atom in [bond.atom1, bond.atom2]]
        if True in is_h: #Assume that there is at most one hydrogen per bond, thus only one element is ever true
            #Obtain Atoms
            h_atom = [bond.atom1, bond.atom2][is_h.index(True)]
            heavy_atom = [bond.atom1, bond.atom2][is_h.index(False)]
            #Indices
            h_ind = h_atom.index
            heavy_ind = heavy_atom.index
            if heavy_ind in not_water_indices:
                covalent_H_bonds.append([heavy_ind, h_ind])
    
    covalent_H_bonds = sorted(covalent_H_bonds)
    
    heavy_order = {}
    for bond in covalent_H_bonds:
        if bond[0] in heavy_order:
            heavy_order[bond[0]] += 1
        else:
            heavy_order[bond[0]] = 1
    
    scale_information = [] #[heavy_ind, hyd_ind, amount to scale H by]
    for bond in covalent_H_bonds:
        scale_information.append([bond[0], bond[1], dH_per_H[heavy_order[bond[0]] - 1]]) #Num H minus 1 gives correct index from previous list
    
    for heavy_ind, hyd_ind, dH in scale_information:
        _ = sys.setParticleMass(hyd_ind, sys.getParticleMass(hyd_ind) + dH)
        _ = sys.setParticleMass(heavy_ind, sys.getParticleMass(heavy_ind) - dH)
    
    #Print Mass after (it should be the same)
    mass_after = sum([sys.getParticleMass(i) for i in range(sys.getNumParticles())])
    print(f"Change in Mass after HMR: {mass_after - mass_before}")
    
    with open(sys_fn.replace('_sys.xml', '_FG_HMR_sys.xml'), 'w') as f:
        f.write(XmlSerializer.serialize(sys))

./Talha_Compounds/Comp1.pdb ./Talha_Compounds/Comp1_sys.xml
<openmm.openmm.HarmonicBondForce; proxy of <Swig Object of type 'OpenMM::HarmonicBondForce *' at 0x1554f33d88f0> > 0
<openmm.openmm.PeriodicTorsionForce; proxy of <Swig Object of type 'OpenMM::PeriodicTorsionForce *' at 0x1554f33d81b0> > 0
<openmm.openmm.NonbondedForce; proxy of <Swig Object of type 'OpenMM::NonbondedForce *' at 0x1554f33d96f0> > 0
<openmm.openmm.CMMotionRemover; proxy of <Swig Object of type 'OpenMM::CMMotionRemover *' at 0x1554f33a9a30> > 0
<openmm.openmm.HarmonicAngleForce; proxy of <Swig Object of type 'OpenMM::HarmonicAngleForce *' at 0x1554f33a90f0> > 0
Change in Mass after HMR: -2.962769940495491e-08 Da
./Talha_Compounds/Comp11.pdb ./Talha_Compounds/Comp11_sys.xml
<openmm.openmm.HarmonicBondForce; proxy of <Swig Object of type 'OpenMM::HarmonicBondForce *' at 0x1554f5c6c830> > 0
<openmm.openmm.PeriodicTorsionForce; proxy of <Swig Object of type 'OpenMM::PeriodicTorsionForce *' at 0x1554f5c6c7b0> > 0
<op